# 09 - Baseline vs Challengers (2 seuils de reference)

Ce notebook lance les entrainements necessaires puis compare les challengers a **deux baselines**
du meme modele de reference (`baseline_logreg`):

1. baseline au seuil fixe `0.50`
2. baseline au seuil optimise `threshold_best_f1`

Modeles compares dans le benchmark:
- `baseline_logreg`
- `lightgbm`
- `stacking_logreg_lgbm`

Sorties attendues:
- `models/challenger_metrics.csv`
- `models/challenger_winner.json`
- modeles calibres baseline/challengers

In [1]:
import os
import pathlib
import sys
import warnings

# Trouver la racine projet de facon robuste (notebook lance depuis notebooks/ ou racine)
cwd = pathlib.Path.cwd().resolve()
if (cwd / 'src').exists() and (cwd / 'requirements.txt').exists():
    ROOT = cwd
elif (cwd.parent / 'src').exists() and (cwd.parent / 'requirements.txt').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError('Impossible de localiser la racine projet contenant src/ et requirements.txt')

# Forcer le dossier de travail a la racine pour des sorties stables (models/, data/, ... )
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings('ignore')
print('Environnement pret:', ROOT)
print('CWD courant:', pathlib.Path.cwd().resolve())

Environnement pret: /workspaces/Brokerflow_ai
CWD courant: /workspaces/Brokerflow_ai


In [2]:
%pip install -q -r {ROOT / 'requirements.txt'} pydantic-settings

import json
from pathlib import Path

import pandas as pd

from src.config.settings import settings
from src.models.train_baseline import save_model, train_baseline_model
from src.models.train_challenger import run_challenger_benchmark

MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 1) (Optionnel) relancer la baseline historique pour garder un point de comparaison local
baseline_pipeline = train_baseline_model(n_samples=1200)
save_model(baseline_pipeline, settings.model_path)
print('Baseline sauvegardée ->', settings.model_path)

2026-03-24 14:10:53 | INFO | src.models.train_baseline | Baseline logistic regression metrics -- AUC: 0.748, Acc: 0.872, Prec: 0.884, Rec: 0.984, F1: 0.931
2026-03-24 14:10:53 | INFO | src.models.train_baseline | Saved baseline model to models/baseline_logreg.pkl
2026-03-24 14:10:53 | INFO | src.models.train_baseline | Saved preprocessor to models/preprocessor.pkl


Baseline sauvegardée -> models/baseline_logreg.pkl


In [4]:
# 2) Lancer le benchmark baseline-challengers (inclut modele d'ensemble)
results, winner = run_challenger_benchmark(n_samples=1500)
results

2026-03-24 14:10:54 | INFO | src.models.train_challenger | Training baseline_logreg
2026-03-24 14:10:54 | INFO | src.models.train_challenger | Saved baseline_logreg to models/baseline_logreg_calibrated.pkl
2026-03-24 14:10:54 | INFO | src.models.train_challenger | Training lightgbm


[LightGBM] [Info] Number of positive: 618, number of negative: 82
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000253 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4020
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.882857 -> initscore=2.019769
[LightGBM] [Info] Start training from score 2.019769
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

2026-03-24 14:10:55 | INFO | src.models.train_challenger | Saved lightgbm to models/lightgbm_calibrated.pkl
2026-03-24 14:10:55 | INFO | src.models.train_challenger | Training stacking_logreg_lgbm


[LightGBM] [Info] Number of positive: 618, number of negative: 82
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000371 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4020
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.882857 -> initscore=2.019769
[LightGBM] [Info] Start training from score 2.019769
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[

2026-03-24 14:11:00 | INFO | src.models.train_challenger | Saved stacking_logreg_lgbm to models/stacking_logreg_lgbm_calibrated.pkl
2026-03-24 14:11:00 | INFO | src.models.train_challenger | Saved comparison table to models/challenger_metrics.csv
2026-03-24 14:11:00 | INFO | src.models.train_challenger | Saved winner metadata to models/challenger_winner.json
2026-03-24 14:11:00 | INFO | src.models.train_challenger | Winner: stacking_logreg_lgbm


,model,roc_auc,brier,threshold_best_f1,accuracy@0.5,precision@0.5,recall@0.5,f1@0.5,accuracy@best,precision@best,recall@best,f1@best,business_score
0,stacking_logreg_lgbm,0.783756,0.096610,0.05,0.880000,0.883669,0.994962,0.936019,0.882222,0.882222,1.000000,0.937426,0.890832
1,baseline_logreg,0.785657,0.093270,0.65,0.882222,0.882222,1.000000,0.937426,0.886667,0.889640,0.994962,0.939358,0.889693
2,lightgbm,0.729861,0.103196,0.05,0.882222,0.882222,1.000000,0.937426,0.882222,0.882222,1.000000,0.937426,0.882418


In [5]:
# 3) Lire les artefacts de comparaison
primary_dir = MODEL_DIR
fallback_dir = Path.cwd() / 'models'

metrics_path = primary_dir / 'challenger_metrics.csv'
winner_path = primary_dir / 'challenger_winner.json'

if (not metrics_path.exists() or not winner_path.exists()) and fallback_dir != primary_dir:
    alt_metrics = fallback_dir / 'challenger_metrics.csv'
    alt_winner = fallback_dir / 'challenger_winner.json'
    if alt_metrics.exists() and alt_winner.exists():
        metrics_path, winner_path = alt_metrics, alt_winner

if not metrics_path.exists() or not winner_path.exists():
    raise FileNotFoundError(
        'Artifacts introuvables. Execute d abord la cellule benchmark (cellule 5), puis relance cette cellule.'
    )

print('Metrics file:', metrics_path)
print('Winner file:', winner_path)

metrics_df = pd.read_csv(metrics_path)
with open(winner_path, 'r', encoding='utf-8') as f:
    winner_payload = json.load(f)

winner_display = winner_payload['winner_model'].replace('champion_logreg', 'baseline_logreg').replace('challenger_lgbm', 'lightgbm')
print('Winner:', winner_display)
print('Rule:', winner_payload['selection_rule'])
print('Best threshold:', winner_payload['winner_threshold_best_f1'])

metrics_df

Metrics file: /workspaces/Brokerflow_ai/models/challenger_metrics.csv
Winner file: /workspaces/Brokerflow_ai/models/challenger_winner.json
Winner: stacking_logreg_lgbm
Rule: 0.45*recall@best + 0.35*f1@best + 0.15*roc_auc - 0.05*brier
Best threshold: 0.05


,model,roc_auc,brier,threshold_best_f1,accuracy@0.5,precision@0.5,recall@0.5,f1@0.5,accuracy@best,precision@best,recall@best,f1@best,business_score
0,stacking_logreg_lgbm,0.783756,0.096610,0.05,0.880000,0.883669,0.994962,0.936019,0.882222,0.882222,1.000000,0.937426,0.890832
1,baseline_logreg,0.785657,0.093270,0.65,0.882222,0.882222,1.000000,0.937426,0.886667,0.889640,0.994962,0.939358,0.889693
2,lightgbm,0.729861,0.103196,0.05,0.882222,0.882222,1.000000,0.937426,0.882222,0.882222,1.000000,0.937426,0.882418


In [6]:
# 4) Construire deux baselines (meme modele, deux seuils) puis comparer
baseline_key = None
for candidate in ['baseline_logreg', 'champion_logreg']:
    if candidate in set(metrics_df['model'].tolist()):
        baseline_key = candidate
        break

if baseline_key is None:
    raise ValueError('Aucune baseline trouvee dans metrics_df (attendu: baseline_logreg ou champion_logreg).')

baseline_row = metrics_df.loc[metrics_df['model'] == baseline_key].iloc[0]

baseline_dual = pd.DataFrame([
    {
        'baseline_type': 'baseline_threshold_0_50',
        'reference_model': baseline_key,
        'threshold_used': 0.50,
        'roc_auc': baseline_row['roc_auc'],
        'brier': baseline_row['brier'],
        'precision': baseline_row['precision@0.5'],
        'recall': baseline_row['recall@0.5'],
        'f1': baseline_row['f1@0.5'],
    },
    {
        'baseline_type': 'baseline_threshold_best',
        'reference_model': baseline_key,
        'threshold_used': baseline_row['threshold_best_f1'],
        'roc_auc': baseline_row['roc_auc'],
        'brier': baseline_row['brier'],
        'precision': baseline_row['precision@best'],
        'recall': baseline_row['recall@best'],
        'f1': baseline_row['f1@best'],
    },
])

baseline_dual

,baseline_type,reference_model,threshold_used,roc_auc,brier,precision,recall,f1
0,baseline_threshold_0_50,baseline_logreg,0.50,0.785657,0.09327,0.882222,1.000000,0.937426
1,baseline_threshold_best,baseline_logreg,0.65,0.785657,0.09327,0.889640,0.994962,0.939358


In [7]:
# 5) Comparaison challengers vs les 2 baselines + conclusion
challengers = metrics_df.loc[metrics_df['model'] != baseline_key, [
    'model', 'threshold_best_f1', 'roc_auc', 'brier',
    'precision@best', 'recall@best', 'f1@best', 'business_score'
 ]].copy()

challengers = challengers.rename(columns={
    'threshold_best_f1': 'challenger_threshold',
    'precision@best': 'challenger_precision',
    'recall@best': 'challenger_recall',
    'f1@best': 'challenger_f1',
})

# Produit cartesien: chaque challenger compare a chaque baseline
comparison = challengers.assign(_k=1).merge(
    baseline_dual.assign(_k=1),
    on='_k',
    how='inner'
).drop(columns=['_k'])

comparison['delta_precision'] = comparison['challenger_precision'] - comparison['precision']
comparison['delta_recall'] = comparison['challenger_recall'] - comparison['recall']
comparison['delta_f1'] = comparison['challenger_f1'] - comparison['f1']

display_cols = [
    'model', 'baseline_type', 'challenger_threshold', 'threshold_used',
    'challenger_precision', 'precision', 'delta_precision',
    'challenger_recall', 'recall', 'delta_recall',
    'challenger_f1', 'f1', 'delta_f1', 'business_score'
 ]
comparison_view = comparison[display_cols].sort_values(
    ['business_score', 'baseline_type'],
    ascending=[False, True]
).reset_index(drop=True)

comparison_view

best_challenger = challengers.sort_values('business_score', ascending=False).iloc[0]
print('Baseline de reference :', baseline_key.replace('champion_logreg', 'baseline_logreg'))
print('Meilleur challenger :', best_challenger['model'])
print(f"F1@best challenger: {best_challenger['challenger_f1']:.4f}")
print(f"Recall@best challenger: {best_challenger['challenger_recall']:.4f}")
print('Decision: comparer ce challenger aux 2 baselines et retenir selon les priorites metier.')

Baseline de reference : baseline_logreg
Meilleur challenger : stacking_logreg_lgbm
F1@best challenger: 0.9374
Recall@best challenger: 1.0000
Decision: comparer ce challenger aux 2 baselines et retenir selon les priorites metier.
